# Starter TF-IDF Ridge Baseline

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from scipy.stats import pearsonr
from sklearn.model_selection import train_test_split
from sklearn.pipeline import FeatureUnion, Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error

SEED = 2026

candidates = [p for p in Path("/kaggle/input").glob("*") if (p/"train.csv").exists() and (p/"test.csv").exists()]
print("Candidates:", candidates)
DATA_DIR = candidates[0]
print("Using:", DATA_DIR)

train = pd.read_csv(DATA_DIR / "train.csv")
test = pd.read_csv(DATA_DIR / "test.csv")

def make_text(df):
    return (
        "anchor: " + df["anchor"].astype(str)
        + " target: " + df["target"].astype(str)
        + " context: " + df["context"].astype(str)
        + " pair: " + df["anchor"].astype(str) + " [SEP] " + df["target"].astype(str)
    )

X = make_text(train)
X_test = make_text(test)
y = train["score"].astype(float).values

tr, va = train_test_split(np.arange(len(train)), test_size=0.2, random_state=SEED, stratify=train["score"])

features = FeatureUnion([
    ("word", TfidfVectorizer(analyzer="word", ngram_range=(1,2), min_df=2, max_features=120000, sublinear_tf=True)),
    ("char", TfidfVectorizer(analyzer="char_wb", ngram_range=(3,5), min_df=2, max_features=160000, sublinear_tf=True)),
])

model = Pipeline([("tfidf", features), ("ridge", Ridge(alpha=8.0, random_state=SEED))])

model.fit(X.iloc[tr], y[tr])
val_pred = np.clip(model.predict(X.iloc[va]), 0, 1)
print("Validation Pearson:", pearsonr(y[va], val_pred)[0])
print("Validation MAE:", mean_absolute_error(y[va], val_pred))

model.fit(X, y)
test_pred = np.clip(model.predict(X_test), 0, 1)

sample = pd.read_csv(DATA_DIR / "sample_submission.csv")
sample["score"] = test_pred
assert sample["id"].astype(str).tolist() == test["id"].astype(str).tolist()
sample.to_csv("/kaggle/working/submission.csv", index=False)
display(sample.head())
print("Saved /kaggle/working/submission.csv")
